# 🍎 FruitTreeScanner 全流程验证

**目标**: 验证 COCO 预训练 YOLOv8s 模型导出 CoreML 并集成到 iOS App 的完整流程

---

**注意**: 这只是验证流程，正式使用需要用自定义水果数据集训练模型

## Step 1: 安装依赖

In [ ]:
!pip install ultralytics -q
print("✅ ultralytics installed")

## Step 2: 加载 YOLOv8s (COCO 预训练)

COCO 包含 apple (class 77) 和 orange (class 78)，可用于快速验证

In [ ]:
from ultralytics import YOLO

# 加载 COCO 预训练模型（包含 apple, orange 等 80 类）
model = YOLO('yolov8s.pt')
print("✅ Model loaded: yolov8s.pt (COCO pretrained)")

# 查看 COCO 中的水果类别
coconut_fruits = {
    77: 'apple',
    78: 'orange',
    39: 'broccoli',
    52: 'banana',
    55: 'orange'  # 另一种橘子
}
print(f"\n🍎 COCO fruit classes available:")
for cid, name in coconut_fruits.items():
    print(f"   Class {cid}: {name}")

## Step 3: 导出 CoreML 模型

In [ ]:
import shutil
from google.colab import files

print("🔄 Exporting to CoreML...")
coreml_path = model.export(format='coreml')
print(f"✅ CoreML model exported to: {coreml_path}")

# 复制到当前目录
shutil.copy(coreml_path, './COCO_FruitsDetector.mlmodel')
print("📁 Model copied to: ./COCO_FruitsDetector.mlmodel")

## Step 4: 验证模型文件

In [ ]:
import os

model_file = './COCO_FruitsDetector.mlmodel'
file_size = os.path.getsize(model_file) / (1024 * 1024)
print(f"📦 Model file: {model_file}")
print(f"💾 Size: {file_size:.2f} MB")

if file_size > 0:
    print("✅ Model file is valid!")
else:
    print("❌ Model file is empty!")

## Step 5: 下载模型

In [ ]:
print("⬇️ 点击下方按钮下载 .mlmodel 文件")
print("   准备好后继续阅读后面的 iOS 集成说明")
files.download('./COCO_FruitsDetector.mlmodel')

---

## iOS 集成说明

### 1. 添加模型到 Xcode 项目

1. 打开 Xcode 项目
2. 将下载的 `COCO_FruitsDetector.mlmodel` 拖入 `FruitTreeScanner` 文件夹
3. 确保 "Target" 勾选了 `FruitTreeScanner`
4. Xcode 会自动编译模型

### 2. 重命名模型（可选）

如果想使用 `FruitsDetector` 这个名字：
```bash
mv COCO_FruitsDetector.mlmodel FruitsDetector.mlmodel
```
（但代码里默认会查找 `FruitsDetector`，所以如果用默认名不需要改代码）

### 3. 更新 FruitModels.swift（重要！）

当前代码只定义了 5 个水果类别，但 COCO 预训练模型输出的是 COCO 类别。
需要添加一个兼容层：

In [ ]:
# 这是代码说明，不需要在 Colab 运行
# 以下代码需要添加到你的 iOS 项目中

/*
// 在 FruitModels.swift 中添加 COCO 映射
// 保留原有 FruitCategory，添加 COCO 兼容映射

enum COCOFruitCategory: Int, CaseIterable {
    case apple = 77
    case orange = 78
    
    var fruitCategory: FruitCategory? {
        switch self {
        case .apple: return .apple
        case .orange: return .orange
        default: return nil
        }
    }
}
*/

print("📝 请查看 FruitModels.swift 中的 COCO 映射说明")

### 4. 当前 ImageDetector 的处理逻辑

ImageDetector 已经支持 CoreML 模型：

```swift
// ImageDetector 会自动：
1. 尝试加载 FruitsDetector.mlmodel
2. 如果找到，使用 VNCoreMLRequest 进行目标检测
3. 如果没找到，回退到 Vision 内置分类器
```

### 5. 测试步骤

1. 运行 App，进入扫描界面
2. 对着苹果或橙子（最好是真水果！）拍摄
3. 点击导出按钮
4. 查看检测结果

**预期结果**：
- 如果水果被正确检测到，计数会包含对应类别
- imageOnly 源的水果会计入 0.5 权重

---

## 下一步：自定义训练

验证流程通过后，下一步是训练自定义水果模型：

1. 收集水果图片（50+/类）
2. 在 Roboflow 标注
3. 运行 `Scripts/train_yolov8.py`
4. 导出 CoreML 并替换

查看详细文档: `docs/FruitDetectionModelGuide.md`